# Conditional Neural Processes (CNP) for 1D regression.
[Conditional Neural Processes](https://arxiv.org/pdf/1807.01613.pdf) (CNPs) were
introduced as a continuation of
[Generative Query Networks](https://deepmind.com/blog/neural-scene-representation-and-rendering/)
(GQN) to extend its training regime to tasks beyond scene rendering, e.g. to
regression and classification.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import datetime
import numpy as np

import os
import plotting_utils_cnp as plotting
import data_generator as data
from matplotlib.backends.backend_pdf import PdfPages
import import_ipynb
import conditional_neural_process_model as cnp
import pickle as pkl
import yaml
from dotenv import load_dotenv
load_dotenv()
resum_path = os.getenv("RESUM_PATH")
resum_lf_tier3_path = os.getenv("RESUM_LF_SIMULATION_TIER3_PATH")


In [4]:
with open("settings.yaml", "r") as f:
    config = yaml.safe_load(f)

TRAINING_ITERATIONS = int(config["cnp_settings"]["training_iterations"]) # Total number of training points: training_iterations * batch_size * max_content_points
MAX_CONTEXT_POINTS = config["cnp_settings"]["max_context_points"]
MAX_TARGET_POINTS =  config["cnp_settings"]["max_target_points"]
CONTEXT_IS_SUBSET = config["cnp_settings"]["context_is_subset"]
BATCH_SIZE = config["cnp_settings"]["batch_size"]
CONFIG_WISE = config["cnp_settings"]["config_wise"]
PLOT_AFTER = int(config["cnp_settings"]["plot_after"])
torch.manual_seed(0)

names_x=config["simulation_settings"]["design_parameters"]
names_x.extend(config["simulation_settings"]["event_specific_parameters"])
x_size = len(names_x)
name_y =config["simulation_settings"]["y_raw_cnp"]

if isinstance(name_y,str):
    y_size = 1
else:
    y_size = len(name_y)

RATIO_TESTING_VS_TRAINING = config["cnp_settings"]["ratio_testing_vs_training"]
version_cnp= config["cnp_settings"]["version"]
version_lf= config["simulation_settings"]["version_lf"]

path_to_files=f"{resum_lf_tier3_path}"
path_out = f'{resum_path}/conditional_neutral_process/out/'
f_out = f'{path_out}CNPGauss_{version_cnp}_{TRAINING_ITERATIONS}_c{MAX_CONTEXT_POINTS}_t{MAX_TARGET_POINTS}'

In [5]:

# Set data augmentation parameters
USE_DATA_AUGMENTATION = config["cnp_settings"]["use_data_augmentation"]
USE_BETA = config["cnp_settings"]["use_beta"]
SIGNAL_TO_BACKGROUND_RATIO = config["cnp_settings"]["signal_to_background_ratio"]

if USE_DATA_AUGMENTATION:
    path_out = f'{resum_path}/out/{USE_DATA_AUGMENTATION}/'
    f_out = f'CNPGauss_{version_cnp}_{TRAINING_ITERATIONS}_c{MAX_CONTEXT_POINTS}_t{MAX_TARGET_POINTS}_{USE_DATA_AUGMENTATION}{SIGNAL_TO_BACKGROUND_RATIO}'
    if USE_DATA_AUGMENTATION == "mixup":
        path_to_files = f"{resum_path}/simulation/out/LF/{version_lf}/tier4/beta_{USE_BETA[0]}_{USE_BETA[1]}/"
        f_out = f'CNPGauss_{version_cnp}_{TRAINING_ITERATIONS}_c{MAX_CONTEXT_POINTS}_t{MAX_TARGET_POINTS}_beta_{USE_BETA[0]}_{USE_BETA[1]}'
    elif USE_DATA_AUGMENTATION == "smote" and CONFIG_WISE == True:
        path_to_files = f"{resum_path}/simulation/out/LF/{version_lf}/tier4/smote{SIGNAL_TO_BACKGROUND_RATIO}/"
        

In [49]:
# Train dataset
dataset_train = data.DataGeneration(num_iterations=TRAINING_ITERATIONS, num_context_points=MAX_CONTEXT_POINTS, num_target_points=MAX_TARGET_POINTS, batch_size = BATCH_SIZE, config_wise=CONFIG_WISE, path_to_files=path_to_files,x_size=x_size,y_size=y_size, mode = "training", ratio_testing=RATIO_TESTING_VS_TRAINING,sig_bkg_ratio = SIGNAL_TO_BACKGROUND_RATIO, use_data_augmentation=USE_DATA_AUGMENTATION, names_x = names_x, name_y=name_y)
TRAINING_ITERATIONS = dataset_train._num_iterations
# Testing dataset
dataset_testing = data.DataGeneration(num_iterations=int(np.round(TRAINING_ITERATIONS/PLOT_AFTER))+5, num_context_points=MAX_CONTEXT_POINTS, num_target_points=MAX_TARGET_POINTS, batch_size = 1, config_wise=False, path_to_files=f"{resum_path}/simulation/out/LF/{version_lf}/tier2/",x_size=x_size,y_size=y_size, mode = "testing",ratio_testing=RATIO_TESTING_VS_TRAINING, sig_bkg_ratio = SIGNAL_TO_BACKGROUND_RATIO, use_data_augmentation="None", names_x = names_x, name_y=name_y)
TRAINING_ITERATIONS = dataset_train._num_iterations if TRAINING_ITERATIONS > dataset_train._num_iterations else TRAINING_ITERATIONS
PLOT_AFTER =  int(5 * np.ceil(np.ceil(TRAINING_ITERATIONS/(dataset_testing._num_iterations-2))/5)) if PLOT_AFTER < int(np.ceil(TRAINING_ITERATIONS/(dataset_testing._num_iterations-2))) else PLOT_AFTER


SystemError: Empty filelist. No neutron tier files found. Run tier file generation first.

We can now add the model to the graph and finalise it by defining the train step
and the initializer.

In [ ]:

d_x, d_in, representation_size, d_out = x_size , x_size+y_size, 32, y_size+1
encoder_sizes = [d_in, 32, 64, 128, 128, 128, 64, 48, representation_size]
decoder_sizes = [representation_size + d_x, 32, 64, 128, 128, 128, 64, 48, d_out]

model = cnp.DeterministicModel(encoder_sizes, decoder_sizes)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
#optimizer = torch.optim.SGD(model.parameters(), lr=0.0001, momentum=0.9)
# 

bce = nn.BCELoss()
iter_testing = 0
fout = open(f'{path_out}{f_out}_training.txt', "w")

# create a PdfPages object
pdf = PdfPages(f'{path_out}{f_out}_training.pdf')

for it in range(TRAINING_ITERATIONS):
    # load data:
    data_train = dataset_train.get_data(it, CONTEXT_IS_SUBSET)

    # Get the predicted mean and variance at the target points for the testing set
    log_prob, mu, _ = model(data_train.query, data_train.target_y)
    
    # Define the loss
    loss = -log_prob.mean()
    loss.backward()

    # Perform gradient descent to update parameters
    optimizer.step()
    
    # reset gradient to 0 on all parameters
    optimizer.zero_grad()

    if max(mu[0].detach().numpy()) <= 1 and min(mu[0].detach().numpy()) >= 0:
        loss_bce = bce(mu, data_train.target_y)
    else:
        loss_bce = -1.

    mu=mu[0].detach().numpy()
    if it % 500 == 0 or it > 3400:
        print('{} Iteration: {}/{}, train loss: {:.4f} (vs BCE {:.4f})'.format(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),it, TRAINING_ITERATIONS,loss, loss_bce))
        fout.write('Iteration: {}/{}, train loss: {:.4f} (vs BCE {:.4f})\n'.format(it, TRAINING_ITERATIONS,loss, loss_bce))
    
    if it % PLOT_AFTER == 0 or it == int(TRAINING_ITERATIONS-1) or it > 3500:
        data_testing = dataset_testing.get_data(iter_testing, CONTEXT_IS_SUBSET)
        log_prob_testing, mu_testing, _ = model(data_testing.query, data_testing.target_y)
        # Define the loss
        loss_testing = -log_prob_testing.mean()

        if max(mu_testing[0].detach().numpy()) <= 1 and min(mu_testing[0].detach().numpy()) >= 0:
            loss_bce_testing = bce(mu_testing,  data_testing.target_y)
        else:
            loss_bce_testing = -1.

        mu_testing=mu_testing[0].detach().numpy()
        print("{}, Iteration: {}, test loss: {:.4f} (vs BCE {:.4f})".format(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), it, loss_testing, loss_bce_testing))
        fout.write("{}, Iteration: {}, test loss: {:.4f} (vs BCE {:.4f})\n".format(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), it, loss_testing, loss_bce_testing))
        if isinstance(name_y,str):
            fig = plotting.plot(mu, data_train.target_y[0].detach().numpy(), f'{loss:.2f}', mu_testing, data_testing.target_y[0].detach().numpy(), f'{loss_testing:.2f}', it)
        else:
            for k in range(y_size):
                fig = plotting.plot(mu[:,k], data_train.target_y[0].detach().numpy()[:,k], f'{loss:.2f}', mu_testing[:,k], data_testing.target_y[0].detach().numpy()[:,k], f'{loss_testing:.2f}', it)
        #if it % PLOT_AFTER*5 == 0 or it == int(TRAINING_ITERATIONS-1) or it > 3500:
        if it % PLOT_AFTER*5 == 0 or it == int(TRAINING_ITERATIONS-1):
            pdf.savefig(fig)
            pkl.dump( fig,  open(f'{resum_path}/out/{f_out}_distr.p',  'wb')  )
            plt.show()
            plt.clf()
        iter_testing += 1
pdf.close()
fout.close()
torch.save(model.state_dict(), f'{resum_path}/out/{f_out}_model.pth')

